# JuaKazi `ha-bias-classifier-v1` — Training Notebook

**Model:** `juakazike/ha-bias-classifier-v1`  
**Base:** `Davlan/afro-xlmr-base` · **Task:** Hausa gender bias (binary)  
**Runtime:** GPU T4 · ~2 hours  
**Target:** BIAS Precision ≥ 0.60, Recall ≥ 0.75, F1 ≥ 0.70

### Data sources (combined, ~27K unique rows)

| File | Rows | Notes |
|---|---|---|
| `v4_revised_hausa_bias_ds.csv` | 17,401 | neutral + stereotype + derogation + counter-stereotype |
| `study-labs-non-synthetics-qa-approved-sentences-*.csv` | 10,178 | all accepted, richer labels |
| `twitter-hausa-bias-*.csv` | 1,608 | social media domain supplement |

**Label mapping:**
- `stereotype` + `derogation` → **BIASED (1)**
- `neutral` → **NEUTRAL (0)**
- `counter-stereotype` → **NEUTRAL (0)** (same as SW v3 — advocacy ≠ bias)

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload all 3 CSV files to Drive at `MyDrive/juakazi/`
3. Add `HF_TOKEN` to Colab Secrets (left sidebar → key icon)


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'transformers>=4.38.0',
    'tokenizers>=0.15.0',
    'accelerate>=0.27.0',
    'scikit-learn>=1.4.0',
    'huggingface_hub>=0.20.0',
    'numpy<2.0.0',
    'datasets>=2.19.0',
], capture_output=True, text=True)

if result.returncode != 0:
    print('INSTALL FAILED:\n', result.stderr)
else:
    print('Install OK. Restart runtime, then run from Cell 2.')

In [ ]:
# ── Cell 2: Verify environment (run AFTER restart) ───────────────────────────
import torch, transformers, sklearn, numpy as np

print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'sklearn:      {sklearn.__version__}')
print(f'numpy:        {np.__version__}')
print(f'GPU:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch to T4!"}')
print(f'CUDA:         {torch.cuda.is_available()}')

assert torch.cuda.is_available(), 'No GPU — go to Runtime → Change runtime type → T4 GPU'

In [ ]:
# ── Cell 3: Mount Drive + load all HA data sources ──────────────────────────
# Kaggle: files are at /kaggle/input/{dataset-slug}/
# Upload your dataset at https://www.kaggle.com/datasets/new
# Add it to this notebook: Data → Add Data → Your datasets

import os, csv

BASE_DIR = '/kaggle/input/juakazi-ha-training'

V4_CSV       = f'{BASE_DIR}/v4_revised_hausa_bias_ds.csv'
SL_CSV       = f'{BASE_DIR}/study-labs-non-synthetics-qa-approved-sentences-2026-04-27T09-23-46-234Z.csv'
TW_CSV       = f'{BASE_DIR}/twitter-hausa-bias-2026-04-29T17-14-06-511Z_classified.csv'

for p in [V4_CSV, SL_CSV]:
    assert os.path.exists(p), f'Not found: {p}\nUpload to Drive first.'

def read_csv(path):
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))

# Load v4 — primary source (has neutral rows)
v4_rows = read_csv(V4_CSV)
# Load StudyLabs — all accepted
sl_rows = read_csv(SL_CSV)
# Load Twitter — supplement for social media domain
tw_rows = read_csv(TW_CSV) if os.path.exists(TW_CSV) else []

print(f'v4:         {len(v4_rows):,} rows')
print(f'StudyLabs:  {len(sl_rows):,} rows')
print(f'Twitter:    {len(tw_rows):,} rows')

In [ ]:
# ── Cell 4: Label mapping + merge + dedup ────────────────────────────────────
from collections import Counter

BIASED_LABELS  = {'stereotype', 'derogation'}
NEUTRAL_LABELS = {'neutral', 'counter-stereotype'}

def get_label_v4(r):
    bl = r.get('bias_label', '').lower().strip()
    if bl in BIASED_LABELS:  return 1
    if bl in NEUTRAL_LABELS: return 0
    return None  # skip unknown

def get_label_sl(r):
    bl = r.get('bias_label', '').lower().strip()
    if bl in BIASED_LABELS:  return 1
    if bl in NEUTRAL_LABELS: return 0
    return None

def get_label_tw(r):
    bc = r.get('bias_category', '').strip()
    if bc == 'Gender':  return 1
    if bc == 'NoBias':  return 0
    return None

all_data = []
seen_texts = set()

for row, get_label in [(v4_rows, get_label_v4), (sl_rows, get_label_sl), (tw_rows, get_label_tw)]:
    for r in row:
        text  = r.get('text', r.get('sentence', '')).strip()
        label = get_label(r)
        if not text or label is None:
            continue
        if text in seen_texts:
            continue
        seen_texts.add(text)
        all_data.append((text, label))

bias_data    = [(t, l) for t, l in all_data if l == 1]
neutral_data = [(t, l) for t, l in all_data if l == 0]

print(f'Total unique rows: {len(all_data):,}')
print(f'Biased:  {len(bias_data):,}')
print(f'Neutral: {len(neutral_data):,}')
print(f'Raw ratio: 1:{len(neutral_data)//max(len(bias_data),1)}')

In [ ]:
# ── Cell 5: Config + seeds ───────────────────────────────────────────────────
import random
import numpy as np
from pathlib import Path

SEED          = 42
BASE_MODEL    = 'Davlan/afro-xlmr-base'
OUTPUT_DIR    = '/kaggle/working/output'
MAX_LEN       = 128

# Class balance: ~13K biased / ~14K neutral → ratio ~1:1
# POS_WEIGHT: cap at 15x regardless (SW v2 lesson — never exceed this)
n_bias    = len(bias_data)
n_neutral = len(neutral_data)
raw_ratio = n_neutral / max(n_bias, 1)
POS_WEIGHT    = min(raw_ratio, 15.0)

# NEUTRAL_RATIO: use all available neutral rows (unlike SW which was 50:1)
NEUTRAL_RATIO = min(int(n_neutral / max(n_bias, 1)) + 1, 10)

BIAS_TARGET   = max(n_bias, 3000)   # augment up to 3K if we have fewer
TRAIN_SPLIT   = 0.80
VAL_SPLIT     = 0.10
# Remaining 10% = test set (held out, not used during training)
EPOCHS        = 10
BATCH         = 16
LR            = 2e-5
WARMUP_RATIO  = 0.10
WEIGHT_DECAY  = 0.01
FREEZE_LAYERS = 6
REPO_ID       = 'juakazike/ha-bias-classifier-v1'

random.seed(SEED)
np.random.seed(SEED)
import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'n_bias={n_bias:,}  n_neutral={n_neutral:,}  raw_ratio={raw_ratio:.1f}x')
print(f'POS_WEIGHT={POS_WEIGHT:.1f}  NEUTRAL_RATIO={NEUTRAL_RATIO}  BIAS_TARGET={BIAS_TARGET}')
print(f'REPO={REPO_ID}')

In [ ]:
# ── Cell 6: Augmentation (Hausa-specific) ────────────────────────────────────
import random

# Common Hausa gender terms — swap to augment biased examples
HA_SYNONYMS = {
    'likita': 'mai kiwon lafiya',
    'mace':   'mutum',
    'namiji': 'mutum',
    'budurwa': 'yarinya',
    'mata':    'jama\'a',
    'matan':   'mutanen',
    'uwargida': 'mai gida',
    'shugaban': 'jagoran',
    'shugabanci': 'jagora',
}

def augment_ha(text: str) -> str:
    words = text.split()
    # Random synonym swap
    if random.random() < 0.5:
        for i, w in enumerate(words):
            lw = w.lower().rstrip('.,;?!')
            if lw in HA_SYNONYMS:
                words[i] = HA_SYNONYMS[lw]
                break
    # Random word drop (preserves bias signal)
    if random.random() < 0.3 and len(words) > 5:
        words.pop(random.randint(1, len(words) - 2))
    return ' '.join(words)

# Smoke test
sample = bias_data[0][0]
print('Original: ', sample[:80])
print('Augmented:', augment_ha(sample)[:80])

In [ ]:
# ── Cell 7: Build train/val/test splits ──────────────────────────────────────
import random

random.shuffle(bias_data)
random.shuffle(neutral_data)

# Augment bias class if below target
augmented = []
bias_texts = [t for t, _ in bias_data]
while len(bias_data) + len(augmented) < BIAS_TARGET:
    src = random.choice(bias_texts)
    augmented.append((augment_ha(src), 1))

all_bias    = bias_data + augmented
neutral_cap = len(all_bias) * NEUTRAL_RATIO
neutral_use = neutral_data[:neutral_cap]

combined = all_bias + neutral_use
random.shuffle(combined)

n       = len(combined)
n_train = int(n * TRAIN_SPLIT)
n_val   = int(n * VAL_SPLIT)

train_data = combined[:n_train]
val_data   = combined[n_train:n_train + n_val]
test_data  = combined[n_train + n_val:]   # held out — only used in Cell 14

print(f'Bias (orig + aug): {len(all_bias):,}')
print(f'Neutral (used):    {len(neutral_use):,}')
print(f'Train: {len(train_data):,}  ({sum(1 for _,l in train_data if l==1):,} bias)')
print(f'Val:   {len(val_data):,}  ({sum(1 for _,l in val_data   if l==1):,} bias)')
print(f'Test:  {len(test_data):,}  ({sum(1 for _,l in test_data  if l==1):,} bias)  ← held out')

In [ ]:
# ── Cell 8: Tokenizer + length check ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

sample_texts = [t for t, _ in bias_data[:500]]
lengths = [len(tokenizer(t, truncation=False)['input_ids']) for t in sample_texts]

print(f'Token lengths — min:{min(lengths)}  median:{int(np.median(lengths))}  '
      f'p95:{int(np.percentile(lengths, 95))}  max:{max(lengths)}')
print(f'MAX_LEN={MAX_LEN} covers p95: {sum(l <= MAX_LEN for l in lengths)/len(lengths)*100:.1f}%')

plt.figure(figsize=(9, 3))
plt.hist(lengths, bins=40, color='steelblue', edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'max_len={MAX_LEN}')
plt.title('Token length distribution (HA bias rows)')
plt.xlabel('Tokens'); plt.ylabel('Count'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 9: Dataset class ────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset

class BiasDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(label, dtype=torch.long),
        }

train_ds = BiasDataset(train_data, tokenizer, MAX_LEN)
val_ds   = BiasDataset(val_data,   tokenizer, MAX_LEN)
test_ds  = BiasDataset(test_data,  tokenizer, MAX_LEN)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
print(f'Sample shape: {train_ds[0]["input_ids"].shape}')

In [ ]:
# ── Cell 10: Model + freeze bottom layers ────────────────────────────────────
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'NEUTRAL', 1: 'BIAS'},
    label2id={'NEUTRAL': 0, 'BIAS': 1},
    ignore_mismatched_sizes=True,
)

frozen = 0
for i, layer in enumerate(model.roberta.encoder.layer):
    if i < FREEZE_LAYERS:
        for p in layer.parameters():
            p.requires_grad = False
        frozen += 1

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Frozen: {frozen} layers  |  Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M params')
print(f'POS_WEIGHT={POS_WEIGHT:.2f}  (raw ratio was {raw_ratio:.1f}x — capped at 15x)')

In [ ]:
# ── Cell 11: WeightedTrainer + metrics ──────────────────────────────────────
import torch
import numpy as np
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        weight  = torch.tensor([1.0, POS_WEIGHT], dtype=torch.float, device=logits.device)
        loss    = torch.nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = int(((preds==1)&(labels==1)).sum())
    fp = int(((preds==1)&(labels==0)).sum())
    fn = int(((preds==0)&(labels==1)).sum())
    p  = tp/(tp+fp) if (tp+fp)>0 else 0
    r  = tp/(tp+fn) if (tp+fn)>0 else 0
    f  = 2*p*r/(p+r) if (p+r)>0 else 0
    print(f'  TP={tp} FP={fp} FN={fn} | P={p:.3f} R={r:.3f} F1={f:.3f}')
    return {'f1': f, 'precision': p, 'recall': r}

print('WeightedTrainer ready.')

In [ ]:
# ── Cell 12: Training args ───────────────────────────────────────────────────
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print('Trainer ready.')

In [ ]:
# ── Cell 13: TRAIN ──────────────────────────────────────────────────────────
result = trainer.train()
print(f'\nDone. Steps={result.global_step}  Train loss={result.training_loss:.4f}')

In [ ]:
# ── Cell 14: Evaluate on TEST SET (held out) ─────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

pred_out = trainer.predict(test_ds)
preds    = np.argmax(pred_out.predictions, axis=-1)
labels   = pred_out.label_ids

print('=== TEST SET RESULTS ===')
print(classification_report(labels, preds, target_names=['NEUTRAL', 'BIAS']))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NEUTRAL','BIAS'], yticklabels=['NEUTRAL','BIAS'])
plt.title('Confusion Matrix (Test Set)'); plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

final_f1   = f1_score(labels, preds, average='binary')
target_met = final_f1 >= 0.70
print(f'\nTest F1: {final_f1:.4f}  (target ≥0.70)  {"TARGET MET ✓" if target_met else "BELOW TARGET — check error analysis"}')

if not target_met:
    print('\nNext steps if below target:')
    print('  1. Reduce POS_WEIGHT (lower precision, higher recall)')
    print('  2. Add more neutral rows from CC-100 HA')
    print('  3. Check NEUTRAL_RATIO — may need to increase')

In [ ]:
# ── Cell 15: Optimal threshold ───────────────────────────────────────────────
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score as sk_f1

logits_t = torch.tensor(pred_out.predictions)
probs    = F.softmax(logits_t, dim=-1)[:, 1].numpy()

bias_p    = probs[labels == 1]
neutral_p = probs[labels == 0]

plt.figure(figsize=(10, 4))
plt.hist(neutral_p, bins=50, alpha=0.6, color='blue',  label='True NEUTRAL', density=True)
plt.hist(bias_p,    bins=50, alpha=0.6, color='red',   label='True BIAS',    density=True)
plt.title('Score distribution'); plt.xlabel('P(BIAS)'); plt.legend()
plt.tight_layout(); plt.show()

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.20, 0.90, 0.01):
    f = sk_f1(labels, (probs >= t).astype(int), average='binary')
    if f > best_f1:
        best_f1, best_t = f, float(t)

print(f'Optimal threshold: {best_t:.2f}  →  F1={best_f1:.4f}')
print(f'\n>>> Set in production:  JUAKAZI_HA_THRESHOLD={best_t:.2f}')

In [ ]:
# ── Cell 16: Error analysis ──────────────────────────────────────────────────
test_texts = [t for t, _ in test_data]
test_labels = [l for _, l in test_data]

opt_preds = (probs >= best_t).astype(int)

fns = [test_texts[i] for i in range(len(test_data)) if test_labels[i]==1 and opt_preds[i]==0]
fps = [test_texts[i] for i in range(len(test_data)) if test_labels[i]==0 and opt_preds[i]==1]

print(f'False Negatives (bias missed):     {len(fns)}')
print(f'False Positives (neutral flagged): {len(fps)}')

print('\n--- Top 20 False Negatives (bias we missed) ---')
for t in fns[:20]:
    print(f'  {t[:100]}')

print('\n--- Top 20 False Positives (neutral we flagged) ---')
for t in fps[:20]:
    print(f'  {t[:100]}')

In [ ]:
# ── Cell 17: Save model + metadata ──────────────────────────────────────────
import json
import numpy as np
from sklearn.metrics import precision_score, recall_score

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

opt_preds_arr = (probs >= best_t).astype(int)
meta = {
    'model_id':      REPO_ID,
    'base_model':    BASE_MODEL,
    'language':      'ha',
    'data_sources':  ['v4_revised_hausa_bias_ds.csv', 'study-labs-*.csv', 'twitter-hausa-bias-*.csv'],
    'train_size':    len(train_data),
    'val_size':      len(val_data),
    'test_size':     len(test_data),
    'n_bias':        n_bias,
    'n_neutral':     n_neutral,
    'pos_weight':    POS_WEIGHT,
    'neutral_ratio': NEUTRAL_RATIO,
    'bias_augmented': BIAS_TARGET,
    'frozen_layers': FREEZE_LAYERS,
    'epochs':        EPOCHS,
    'lr':            LR,
    'threshold':     best_t,
    'test_f1':       round(best_f1, 4),
    'test_precision': round(float(precision_score(labels, opt_preds_arr, zero_division=0)), 4),
    'test_recall':   round(float(recall_score(labels, opt_preds_arr, zero_division=0)), 4),
    'target_met':    best_f1 >= 0.70,
    'id2label':      {0: 'NEUTRAL', 1: 'BIAS'},
}
with open(f'{OUTPUT_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to:', OUTPUT_DIR)
print(json.dumps(meta, indent=2))

if not meta['target_met']:
    print('\n⚠️  TARGET NOT MET. Do NOT upload to HF until target is met.')
    print('See Cell 14 error analysis for next steps.')
else:
    print('\n✓ Target met. Proceed to Cell 18 to upload to HuggingFace.')

In [ ]:
# ── Cell 18: Integration smoke test (run before uploading) ───────────────────
import torch
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline(
    'text-classification',
    model=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=MAX_LEN,
)

# Known biased HA sentences
tests = [
    ('Likitan namiji ne kawai zai iya jagorantar asibiti.',      True),
    ('Mata ba sa iya jagorantar kasuwanci kamar maza.',          True),
    ('Mace ta fi dacewa da dafa abinci fiye da jagorantar hukuma.', True),
    ('Wata mace ta lashe kyautar bincike ta kasa.',               False),
    ('An gudanar da zabe a jihar Kano jiya.',                    False),
    ('Tattalin arzikin Najeriya ya nuna ci gaba a wannan shekarar.', False),
]

print(f'{"Expected":8} {"Label":10} {"Score":6}  {"Pass":4}  Text')
print('-' * 80)
passed = 0
for text, expected in tests:
    r      = pipe(text)[0]
    label  = r['label'].upper()
    score  = r['score']
    p_bias = score if label == 'BIAS' else 1.0 - score
    predicted = p_bias >= best_t
    ok     = predicted == expected
    if ok: passed += 1
    mark = 'OK' if ok else 'FAIL'
    print(f'{"BIAS" if expected else "NEU":8} {label:10} {p_bias:.3f}  {mark}   {text[:60]}')

print(f'\nResult: {passed}/{len(tests)}  {"ALL PASSED" if passed==len(tests) else "REVIEW FAILURES"}')

In [ ]:
# ── Cell 19: Upload to HuggingFace ──────────────────────────────────────────
# Only run this cell if Cell 17 shows target_met=True
from huggingface_hub import HfApi
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    print('HF_TOKEN not found. Add it in Colab Secrets (left sidebar → key icon).')
    HF_TOKEN = None

if HF_TOKEN and meta['target_met']:
    api = HfApi()
    api.create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True, private=False)

    # Write model card
    model_card = f'''---
language:
- ha
license: apache-2.0
tags:
- text-classification
- gender-bias
- hausa
- afro-xlmr
- west-africa
metrics:
- f1
- precision
- recall
base_model: Davlan/afro-xlmr-base
---

# JuaKazi Hausa Gender Bias Classifier v1

Fine-tuned [afro-xlmr-base](https://huggingface.co/Davlan/afro-xlmr-base) for binary gender bias detection in Hausa text.

Part of the [JuaKazi Gender Sensitization Engine](https://huggingface.co/spaces/juakazike/gender-sensitization-engine).

## Training Data

| Source | Rows |
|---|---|
| v4_revised_hausa_bias_ds.csv | 17,401 |
| study-labs non-synthetics (accepted) | 10,178 |
| twitter-hausa-bias | 1,608 |

Labels: stereotype + derogation → BIASED, neutral + counter-stereotype → NEUTRAL

## Test Metrics

| Metric | Value |
|---|---|
| BIAS Precision | {meta["test_precision"]} |
| BIAS Recall | {meta["test_recall"]} |
| BIAS F1 | {meta["test_f1"]} |
| Decision threshold | {meta["threshold"]} |

## Usage

```python
from transformers import pipeline
pipe = pipeline("text-classification", model="{REPO_ID}")
pipe("Likitan namiji ne kawai zai iya jagorantar asibiti.")
# [{{"label": "BIAS", "score": 0.95}}]
```
'''
    card_path = f'{OUTPUT_DIR}/README.md'
    with open(card_path, 'w') as f:
        f.write(model_card)

    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, token=HF_TOKEN)
    print(f'\nUploaded → https://huggingface.co/{REPO_ID}')
    print(f'\nNext: wire into pipeline:')
    print(f'  JUAKAZI_HA_MODEL={REPO_ID}')
    print(f'  JUAKAZI_HA_THRESHOLD={best_t:.2f}')
elif not meta.get('target_met'):
    print('⚠️  Not uploading — target not met. Re-run training with adjusted POS_WEIGHT.')
else:
    print('Set HF_TOKEN in Colab Secrets to upload.')